# Training the NumPy Seq2Seq + Bahdanau Attention Summarizer

This notebook is just the **training** part of the project pulled out of
`train.py` so it can be run cell-by-cell (locally in Jupyter/VS Code, or in
Google Colab).

**Running in Google Colab:**
1. Upload *this notebook* to Colab (File -> Upload notebook), or open it
   directly from GitHub (File -> Open notebook -> GitHub -> paste the repo
   URL and pick `numpy_seq2seq/train.ipynb`).
2. Run the cell below - it detects Colab and clones
   `github.com/knah1d/seq2seq` (a public repo, so no login needed), which
   includes the dataset, then switches into `numpy_seq2seq/`. Re-running
   the cell later in the same session `git pull`s the latest code instead
   of re-cloning.
3. Runtime type doesn't matter - a GPU gives **no speedup** here since this
   is plain NumPy with no GPU calls; CPU runtime is fine (and free-tier
   friendly).

**Running locally instead:** just open this notebook from inside the
`numpy_seq2seq/` folder (VS Code / Jupyter) - the cell below detects it is
not Colab and does nothing extra.

**Note either way:** NumPy is the model's only dependency (`matplotlib` is
used below just to plot the loss curve).

In [ ]:
import os, sys, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/knah1d/seq2seq.git"
REPO_DIR = "seq2seq"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print("Repo already cloned, pulling latest changes...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    os.chdir(os.path.join(REPO_DIR, "numpy_seq2seq"))
    print("Running from:", os.getcwd())

sys.path.insert(0, os.getcwd())

import time
import numpy as np

from data import prepare_dataset
from model import Seq2SeqAttention
from optim import Adam, clip_grads_

# Evaluation helpers are reused from train.py so the notebook and the CLI
# script cannot drift apart. (Importing train.py is safe - its main() only
# runs under `if __name__ == "__main__"`.)
from train import evaluate, evaluate_rouge, lead_baseline_rouge, ids_to_text

In [ ]:
# Hyperparameters (same defaults as train.py --help)
VOCAB_SIZE   = 8000
ENC_MAX_LEN  = 120   # article tokens the encoder reads (99% of targets land inside this)
DEC_MAX_LEN  = 32    # summary tokens (p90 key-sentence length is 29, + <sos>/<eos>)
EMB_DIM      = 96
HIDDEN_SIZE  = 128
BATCH_SIZE   = 32
EPOCHS       = 80    # upper bound - early stopping will almost certainly stop sooner
LR           = 1e-3
CLIP_NORM    = 5.0
DROPOUT      = 0.3   # regularization: ~3M params on ~6k examples memorizes without it
WEIGHT_DECAY = 1e-5
SEED         = 0
PATIENCE           = 6   # stop after this many epochs with no val ROUGE-L improvement
LR_DECAY_PATIENCE  = 2   # halve the LR after this many epochs with no improvement
CHECKPOINT_PATH    = "checkpoint.npz"

## 1. Load and preprocess the dataset

The decoder target is **not** a truncation of the raw BBC summary. Those summaries are
*extractive* (real article sentences copied verbatim but **reordered**), so a fixed
truncation grabs text from anywhere in the article: measured over all 2225 pairs, 61% of
the time the summary's first sentence starts beyond the encoder's window, and only 6% of
truncated targets ended at a sentence boundary - so the model never learned where to stop.

Instead each article contributes one example per summary sentence that actually appears
**inside** the encoder's input window. Every target is therefore a complete, human-selected,
reachable sentence - and we get ~3x more training data out of the same dataset.

In [ ]:
ds = prepare_dataset(vocab_size=VOCAB_SIZE, enc_max_len=ENC_MAX_LEN, dec_max_len=DEC_MAX_LEN)
print(f"train examples: {len(ds['enc_ids_train'])}, val examples: {len(ds['enc_ids_val'])}")
print(f"vocab size: {len(ds['itos'])}\n")

# Eyeball a couple of (article, target) pairs.
for article, target in ds["raw_val"][:2]:
    print("ARTICLE:", " ".join(article[:25]), "...")
    print("TARGET :", " ".join(target), "\n")

## 2. Build the model and optimizer

In [ ]:
model = Seq2SeqAttention(
    vocab_size=len(ds["itos"]), emb_dim=EMB_DIM,
    hidden_size=HIDDEN_SIZE, dropout=DROPOUT, seed=SEED,
)
optimizer = Adam(model.params, lr=LR, weight_decay=WEIGHT_DECAY)
rng = np.random.RandomState(SEED)

n_params = sum(p.size for p in model.params.values())
print(f"parameters: {n_params:,}")

In [ ]:
def iterate_batches(enc_ids, dec_ids, batch_size, rng, shuffle=True):
    """Yield shuffled mini-batches of (encoder ids, decoder ids)."""
    n = enc_ids.shape[0]
    order = rng.permutation(n) if shuffle else np.arange(n)
    for start in range(0, n, batch_size):
        idx = order[start:start + batch_size]
        yield enc_ids[idx], dec_ids[idx]

## 3. Training loop

Each batch: forward (encoder -> attention -> decoder) -> manual backward (BPTT) ->
gradient clipping -> Adam step.

Each epoch we also greedy-decode the whole validation set and score it with **ROUGE**.
That matters because loss and summary quality diverge once the model starts overfitting:
loss measures per-token confidence, ROUGE measures whether the finished summary actually
overlaps the reference. **The checkpoint is saved on best ROUGE-L, not best loss.**

We also print the **lead-1 baseline** (just emit the article's first sentence) up front.
If the model can't beat that, it hasn't learned anything useful - strong lead baselines
on news summarization are a well-known real result, so this is an honest reference point.

In [ ]:
baseline = lead_baseline_rouge(ds)
print(f"lead-1 baseline: R1={baseline['rouge1']:.3f} R2={baseline['rouge2']:.3f} RL={baseline['rougeL']:.3f}")
print("  -> the model needs to beat this to have learned anything useful.\n")

enc_ids_train, dec_ids_train = ds["enc_ids_train"], ds["dec_ids_train"]

train_loss_history, val_loss_history, rouge_l_history = [], [], []
best_rouge_l = -1.0
epochs_without_improvement = 0
current_lr = LR

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    total_loss_tokens = 0.0
    total_tokens = 0.0
    for enc_batch, dec_batch in iterate_batches(enc_ids_train, dec_ids_train, BATCH_SIZE, rng):
        avg_loss, num_real, cache = model.forward(enc_batch, dec_batch, training=True)
        grads = model.backward(cache)
        clip_grads_(grads, max_norm=CLIP_NORM)
        optimizer.step(model.params, grads)

        total_loss_tokens += avg_loss * num_real
        total_tokens += num_real

    train_loss = total_loss_tokens / total_tokens
    val_loss = evaluate(model, ds["enc_ids_val"], ds["dec_ids_val"], BATCH_SIZE, rng)
    rouge = evaluate_rouge(model, ds, BATCH_SIZE)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    rouge_l_history.append(rouge["rougeL"])

    elapsed = time.time() - epoch_start
    print(f"epoch {epoch}/{EPOCHS} - train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
          f"R1={rouge['rouge1']:.3f} R2={rouge['rouge2']:.3f} RL={rouge['rougeL']:.3f} "
          f"| lr={current_lr:.2e} ({elapsed:.1f}s)")

    if rouge["rougeL"] > best_rouge_l + 1e-4:
        best_rouge_l = rouge["rougeL"]
        epochs_without_improvement = 0
        model.save(CHECKPOINT_PATH)
        print("  -> new best ROUGE-L, saved checkpoint")
    else:
        epochs_without_improvement += 1
        print(f"  -> no ROUGE-L improvement ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement % LR_DECAY_PATIENCE == 0:
            current_lr *= 0.5
            optimizer.set_lr(current_lr)
            print(f"  -> lowered learning rate to {current_lr:.2e}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: no ROUGE-L improvement for {PATIENCE} epochs in a row.")
        break

print(f"\nBest val ROUGE-L={best_rouge_l:.4f} (lead-1 baseline {baseline['rougeL']:.4f})")
print(f"checkpoint saved to {CHECKPOINT_PATH}")

## 4. Plot the curves

Left: if validation loss flattens or rises while training loss keeps falling, that gap
**is** overfitting. Right: ROUGE-L against the lead-1 baseline - this is the number that
actually says whether the summaries are any good.

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(train_loss_history) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_ran, train_loss_history, marker="o", label="train loss")
ax1.plot(epochs_ran, val_loss_history, marker="o", label="val loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy per token")
ax1.set_title("Training vs. validation loss"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, rouge_l_history, marker="o", color="tab:green", label="val ROUGE-L")
ax2.axhline(baseline["rougeL"], ls="--", color="gray", label="lead-1 baseline")
ax2.set_xlabel("epoch"); ax2.set_ylabel("ROUGE-L (F1)")
ax2.set_title("Validation ROUGE-L"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Inspect some generated summaries

`greedy_decode` never emits `<pad>`/`<sos>`/`<unk>` (those logits are masked to -inf) and
blocks immediate word repeats, so what you see below is what the model actually chose.

In [ ]:
pred_ids, _ = model.greedy_decode(ds["enc_ids_val"][:5], max_len=DEC_MAX_LEN)
for i in range(5):
    print("ARTICLE   :", ids_to_text(ds["enc_ids_val"][i], ds["itos"])[:160], "...")
    print("REFERENCE :", ids_to_text(ds["dec_ids_val"][i], ds["itos"]))
    print("PREDICTED :", ids_to_text(pred_ids[i], ds["itos"]))
    print()

## 6. Get the trained model out of Colab

`checkpoint.npz` lives on Colab's temporary disk and **is deleted when the runtime
disconnects**, so save it somewhere before you close the tab.

Option A (below) downloads it straight to your machine (~20MB). Option B copies it to
Google Drive instead, which is more reliable for big files and survives disconnects.

You do **not** need to download `data_cache.npz`: the vocabulary is rebuilt
deterministically from the same files with the same seed, so running
`prepare_dataset()` locally reproduces exactly the same word<->id mapping.

In [ ]:
# --- Option A: download the checkpoint to your computer ---
if IN_COLAB:
    from google.colab import files
    files.download(CHECKPOINT_PATH)

# --- Option B: save to Google Drive instead (uncomment to use) ---
# if IN_COLAB:
#     from google.colab import drive
#     import shutil
#     drive.mount("/content/drive")
#     shutil.copy(CHECKPOINT_PATH, "/content/drive/MyDrive/checkpoint.npz")
#     print("saved to Google Drive as checkpoint.npz")